# Orquestrador de Treino — Risco de Crédito (Lending Club)

Este notebook liga as quatro peças que vivem em `src/` (`data.py`, `models.py`,
`training.py`, `utils.py`) numa execução única e reprodutível: carregamento do
dataset bruto do Kaggle, engenharia de atributos, particionamento temporal,
treino de dois MLPs (classificação de inadimplência e regressão de taxa de
juros), baselines lineares de referência e exportação dos artefatos finais
(checkpoints, figuras, log em markdown, registro de experimentos em JSON).

**Compatível com Google Colab e execução local.** As únicas coisas que
normalmente precisam de ajuste manual antes de rodar são marcadas com
`# AJUSTE` nos comentários — a mais importante é `BASE_DIR`, na seção 2.

Ordem de execução das etapas de dados (não é arbitrária — está documentada
na seção 8): `earliest_cr_line` precisa rodar enquanto `issue_d` ainda existe
no formato original, e o One-Hot Encoding precisa rodar **antes** do split
temporal para as colunas dummy ficarem sincronizadas entre treino/val/teste.

## 1. Ambiente e dependências

Detecta se a execução é no Google Colab (para acionar montagem do Drive e
instalação de pacotes que não vêm pré-instalados) ou local (onde se assume
que o ambiente já tem as dependências do `requirements.txt` do projeto).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Executando no Google Colab: {IN_COLAB}")


In [ ]:
if IN_COLAB:
    # Polars e a CLI do Kaggle não vêm pré-instaladas no runtime padrão do
    # Colab. torch, scikit-learn, matplotlib e tensorboard já vêm prontos.
    # subprocess (em vez de %pip) para não depender de magic dentro de if.
    import subprocess
    import sys as _sys
    subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q", "polars>=1.0.0", "kaggle>=1.5.0"],
        check=True,
    )


## 2. Caminhos e persistência

No Colab, tudo que for salvo em `/content` é perdido quando o runtime
reinicia (timeout de inatividade, crash de memória, etc.). Por isso
montamos o Google Drive e gravamos checkpoints, figuras, logs e o CSV bruto
sempre dentro de `BASE_DIR`, nunca em `/content` diretamente — mesma
convenção já usada no restante do projeto (ver `paths.py` da v3).

**AJUSTE** `BASE_DIR` abaixo para o caminho real deste projeto no seu Drive
(ou, em execução local, deixe `"."` assumindo que o notebook é aberto com o
diretório de trabalho em `projeto_1_mlp/`).

In [ ]:
import os
import sys

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # AJUSTE: caminho da pasta do projeto dentro do seu Drive.
    BASE_DIR = "/content/drive/MyDrive/rede_neural_projetos/projeto_1_mlp"
else:
    # Execução local: assume que o notebook está em projeto_1_mlp/, ao lado de src/.
    BASE_DIR = "."

BASE_DIR = os.path.abspath(BASE_DIR)
print(f"BASE_DIR = {BASE_DIR}")


In [ ]:
# Estrutura de saída — espelha outputs/report_assets já usado como default
# em utils.py e training.py, mas ancorada em BASE_DIR para sobreviver a
# reinícios de runtime no Colab.
DIR_DADOS_RAW = os.path.join(BASE_DIR, "data", "raw")
DIR_OUTPUTS = os.path.join(BASE_DIR, "outputs")
DIR_CHECKPOINTS = os.path.join(DIR_OUTPUTS, "checkpoints")
DIR_REPORT_ASSETS = os.path.join(DIR_OUTPUTS, "report_assets")
DIR_RUNS = os.path.join(BASE_DIR, "runs")  # logs do TensorBoard

for d in (DIR_DADOS_RAW, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS):
    os.makedirs(d, exist_ok=True)

print("Diretórios prontos:")
for d in (DIR_DADOS_RAW, DIR_CHECKPOINTS, DIR_REPORT_ASSETS, DIR_RUNS):
    print(f"  {d}")


## 3. Imports do projeto (`src/`)

Os módulos em `src/` usam imports absolutos no estilo `from src.utils import
...`, então quem precisa estar no `sys.path` é `BASE_DIR` (a pasta que
**contém** `src/`), não `src/` em si.

In [ ]:
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import numpy as np
import polars as pl
import torch
import torch.nn as nn

from src.data import (
    carregar_df_raw,
    tratar_earliest_cr_line,
    agrupar_home_ownership,
    extrair_term_meses,
    mapear_emp_length,
    tratar_dti,
    tratar_annual_inc,
    aplicar_transformacoes_por_regra,
    codificar_categoricas,
    split_temporal,
    tratar_nulos_residuais,
    padronizar_numericas,
    extrair_arrays,
    preparar_dataloaders,
)
from src.models import (
    criar_mlp_classificacao_padrao,
    criar_mlp_regressao_padrao,
    treinar_baseline_classificacao,
    treinar_baseline_regressao,
)
from src.training import (
    fixar_seeds,
    treinar_modelo,
    carregar_melhor_modelo,
    registrar_experimento,
    exportar_experimentos,
)
from src.utils import (
    log_etapa,
    log_nota,
    exportar_log,
    plot_curvas_loss,
    plot_gradient_norm,
    capturar_distribuicao_ativacoes,
    avaliar_classificacao,
    avaliar_regressao,
)

print("Módulos do projeto importados com sucesso.")


## 4. Reprodutibilidade e dispositivo

`fixar_seeds` precisa rodar **antes** de qualquer coisa que consuma
aleatoriedade global — instanciação dos modelos (inicialização de pesos) e
criação dos `DataLoader` de treino (`shuffle=True`). Chamá-la aqui, logo no
início, garante essa ordem.

In [ ]:
SEED = 42
fixar_seeds(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo selecionado: {DEVICE}")

# Identificador único desta execução — evita que checkpoints e runs do
# TensorBoard de execuções diferentes se sobrescrevam silenciosamente
# (treinar_modelo não protege contra isso sozinho).
from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID = {RUN_ID}")


## 5. Aquisição do dataset (Kaggle — Lending Club)

Dataset: [`wordsforthewise/lending-club`](https://www.kaggle.com/datasets/wordsforthewise/lending-club),
arquivo `accepted_2007_to_2018Q4.csv` (mesma fonte já usada nas iterações
anteriores deste projeto).

Requer um token `kaggle.json` (Account → Create New Token, em kaggle.com).
No Colab, faça upload do arquivo quando solicitado; localmente, coloque-o em
`~/.kaggle/kaggle.json`. Se o CSV já existir em `DIR_DADOS_RAW`, o download é
pulado.

In [ ]:
KAGGLE_DATASET = "wordsforthewise/lending-club"
KAGGLE_FILENAME = "accepted_2007_to_2018Q4.csv"
RAW_CSV_PATH = os.path.join(DIR_DADOS_RAW, KAGGLE_FILENAME)

if not os.path.exists(RAW_CSV_PATH) and IN_COLAB:
    kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(kaggle_json):
        print("Faça upload do seu kaggle.json (Account -> Create New Token no Kaggle):")
        from google.colab import files
        uploaded = files.upload()
        os.makedirs(os.path.dirname(kaggle_json), exist_ok=True)
        with open(kaggle_json, "wb") as f:
            f.write(next(iter(uploaded.values())))
        os.chmod(kaggle_json, 0o600)


In [ ]:
if not os.path.exists(RAW_CSV_PATH):
    print(f"Baixando '{KAGGLE_DATASET}' em '{DIR_DADOS_RAW}'...")
    exit_code = os.system(
        f'kaggle datasets download -d {KAGGLE_DATASET} -p "{DIR_DADOS_RAW}" --unzip'
    )
    if exit_code != 0 or not os.path.exists(RAW_CSV_PATH):
        raise FileNotFoundError(
            f"Não foi possível localizar '{RAW_CSV_PATH}' após o download. "
            "Verifique o kaggle.json e o nome do arquivo dentro do dataset."
        )
else:
    print(f"CSV já presente em '{RAW_CSV_PATH}' — download pulado.")


## 6. Carregamento bruto + perfil de nulos

`carregar_df_raw` escaneia o CSV de forma preguiçosa (via `pl.scan_csv`),
mede o percentual de nulos de cada coluna numa única passada e materializa
só as colunas abaixo do limiar — importante aqui porque o CSV completo do
Lending Club tem ~150 colunas e é grande o bastante (alguns GB) para tornar
carregamento eager custoso.

In [ ]:
df_raw = carregar_df_raw(RAW_CSV_PATH, limiar_nulos_pct=50.0)
log_nota(f"df_raw carregado: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
df_raw.head()


## 7. Seleção de colunas de trabalho e criação dos targets

O CSV bruto do Lending Club **não** tem `target_classificacao` nem
`target_regressao` prontos — eles precisam ser derivados de `loan_status`
(inadimplência) e `int_rate` (taxa de juros). Essa derivação, junto com a
lista de colunas curada abaixo, replica a decisão já validada na EDA deste
projeto (`projeto_1_mlp_v3/scr/data_pipeline.py:montar_dataset_bruto`):

- Mantém `id` (rastreabilidade) e `issue_d` (necessário para
  `anos_historico_credito` e para o split temporal, etapa 9).
- Remove desde a origem `grade`/`sub_grade` (vazam `int_rate` quase
  deterministicamente — o Lending Club precifica por sub-grade) e
  `installment` (derivada de `loan_amnt`/`term`/`int_rate`).
- Mantém só empréstimos com desfecho definido: `Current`, `Late` e
  `In Grace Period` não têm rótulo válido de inadimplência ainda.

In [ ]:
COLUNAS_TRABALHO = [
    "id", "issue_d", "loan_amnt", "term", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "addr_state", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "initial_list_status", "application_type", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "collections_12_mths_ex_med", "mort_acc",
    "pub_rec_bankruptcies", "tax_liens", "disbursement_method",
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m",
    "mths_since_rcnt_il", "total_bal_il", "il_util", "open_rv_12m",
    "open_rv_24m", "max_bal_bc", "all_util", "total_rev_hi_lim", "inq_fi",
    "total_cu_tl", "inq_last_12m", "acc_open_past_24mths", "avg_cur_bal",
    "bc_open_to_buy", "bc_util", "chargeoff_within_12_mths", "delinq_amnt",
    "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl", "mths_since_recent_bc", "mths_since_recent_inq",
    "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl",
    "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
    "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m",
    "num_tl_30dpd", "num_tl_90g_dpd_24m", "num_tl_op_past_12m",
    "pct_tl_nvr_dlq", "percent_bc_gt_75", "tot_hi_cred_lim",
    "total_bal_ex_mort", "total_bc_limit", "total_il_high_credit_limit",
    "loan_status", "int_rate",
]

# Algumas colunas curadas podem já ter sido removidas em carregar_df_raw()
# por excesso de nulo (esperado para parte das colunas "il"/"rev" em safras
# mais antigas do dataset) — segue sem quebrar, mas registra o que faltou.
colunas_disponiveis = [c for c in COLUNAS_TRABALHO if c in df_raw.columns]
colunas_faltando = sorted(set(COLUNAS_TRABALHO) - set(colunas_disponiveis))
if colunas_faltando:
    log_nota(
        f"{len(colunas_faltando)} coluna(s) da lista curada não estavam em df_raw "
        f"(removidas pelo filtro de nulos ou ausentes no schema): {colunas_faltando}"
    )


In [ ]:
# Só empréstimos com desfecho definido viram exemplo de treino/avaliação.
STATUS_DESFECHO_DEFINIDO = [
    "Fully Paid", "Charged Off", "Default",
    "Does not meet the credit policy. Status:Fully Paid",
    "Does not meet the credit policy. Status:Charged Off",
]
STATUS_INADIMPLENCIA = [
    "Charged Off", "Does not meet the credit policy. Status:Charged Off", "Default",
]

dist_status = (
    df_raw.group_by("loan_status").len().sort("len", descending=True)
)
log_etapa("Distribuição de loan_status (antes do filtro de desfecho definido)", dist_status)

df_trabalho = (
    df_raw
    .select(colunas_disponiveis)
    .filter(pl.col("loan_status").is_in(STATUS_DESFECHO_DEFINIDO))
    .with_columns(
        pl.col("loan_status").is_in(STATUS_INADIMPLENCIA).cast(pl.Int8).alias("target_classificacao"),
        pl.col("int_rate").alias("target_regressao"),
    )
    .drop(["loan_status", "int_rate"])
)

log_etapa("Dataset de trabalho após seleção de colunas e criação dos targets", f"shape: {df_trabalho.shape}")


## 8. Engenharia de atributos

A ordem das chamadas abaixo importa e não é arbitrária:

1. `tratar_earliest_cr_line` precisa rodar **antes** de qualquer outra coisa
   tocar `issue_d`, porque consome as duas colunas de data no formato bruto
   (`"%b-%Y"`) e descarta `earliest_cr_line` ao final — só sobra
   `anos_historico_credito`, uma feature numérica derivada.
2. As demais transformações específicas (`agrupar_home_ownership` até
   `tratar_annual_inc`) resolvem sentinelas e formatos de colunas
   individuais e não têm dependência de ordem entre si.
3. `aplicar_transformacoes_por_regra` roda **depois** das transformações
   específicas — ela aplica `log1p` a qualquer coluna numérica com skew
   alto, e rodar antes duplicaria a transformação já aplicada manualmente
   em `annual_inc`.
4. `codificar_categoricas` (One-Hot Encoding) roda **antes** do split
   temporal (etapa 9) — assim as colunas dummy são geradas uma única vez
   sobre o dataset inteiro, garantindo que treino/val/teste terminem com
   exatamente as mesmas colunas.

In [ ]:
# 8.1 — anos_historico_credito (substitui earliest_cr_line; issue_d é preservado).
df_trabalho = tratar_earliest_cr_line(df_trabalho)
print(f"shape após tratar_earliest_cr_line: {df_trabalho.shape}")


In [ ]:
# 8.2 — consolida classes raras de home_ownership em 'OTHER'.
df_trabalho = agrupar_home_ownership(df_trabalho)


In [ ]:
# 8.3 — term ('36 months' / '60 months') vira term_meses numérico.
df_trabalho = extrair_term_meses(df_trabalho)


In [ ]:
# 8.4 — emp_length vira escala ordinal + flag de nulo (emp_length_missing).
df_trabalho = mapear_emp_length(df_trabalho)


In [ ]:
# 8.5 — unifica sentinelas de dti (999, -1) em nulo + flag dti_missing.
df_trabalho = tratar_dti(df_trabalho)


In [ ]:
# 8.6 — annual_inc: imputação pela mediana + clipping inferior + log1p.
df_trabalho = tratar_annual_inc(df_trabalho)


In [ ]:
# 8.7 — regra genérica de log1p para as demais colunas numéricas com skew alto.
df_trabalho = aplicar_transformacoes_por_regra(df_trabalho)
print(f"shape após regras de assimetria: {df_trabalho.shape}")


In [ ]:
# 8.8 — One-Hot Encoding das categóricas, ainda sobre o dataset inteiro
# (ver nota da seção 8 sobre por que isso roda antes do split temporal).
df_trabalho = codificar_categoricas(df_trabalho)
log_etapa("Dataset após engenharia de atributos completa", f"shape: {df_trabalho.shape}")


## 9. Particionamento temporal

`split_temporal` ordena por `issue_d` e corta cronologicamente — treino
sempre estritamente anterior a validação, que é anterior a teste. Isso
simula a situação real de produção (prever empréstimos futuros com um
modelo treinado só no passado) e é o motivo de `issue_d` ter sido
preservado intacto na etapa 8.1.

In [ ]:
df_treino, df_val, df_teste = split_temporal(df_trabalho, frac_treino=0.70, frac_val=0.15)

for nome, df_split in [("treino", df_treino), ("validação", df_val), ("teste", df_teste)]:
    datas = df_split["issue_d"].str.to_date("%b-%Y")
    log_nota(
        f"{nome}: {df_split.shape[0]} linhas | issue_d de {datas.min()} até {datas.max()}"
    )


## 10. Nulos residuais e padronização

Ambas as etapas usam **só estatística do treino** (mediana para nulos,
média/desvio-padrão para o z-score) e aplicam o mesmo valor a validação e
teste — evita vazamento de informação dos conjuntos de avaliação para o
pré-processamento.

In [ ]:
df_treino, df_val, df_teste = tratar_nulos_residuais(df_treino, df_val, df_teste)


In [ ]:
df_treino, df_val, df_teste, stats_padronizacao = padronizar_numericas(df_treino, df_val, df_teste)
log_nota(f"{len(stats_padronizacao)} colunas contínuas padronizadas (z-score, estatística de treino).")


## 11. DataLoaders PyTorch

`preparar_dataloaders` valida NaN/Inf/dimensões antes de criar os
`DataLoader` e usa `drop_last=True` nos loaders de treino — necessário
porque os MLPs usam `BatchNorm1d`, que não aceita um batch de tamanho 1
durante o treino (o último batch de uma época pode ter exatamente 1 amostra
se o tamanho do treino não for múltiplo de `batch_size`).

In [ ]:
BATCH_SIZE = 512
dados = preparar_dataloaders(df_treino, df_val, df_teste, batch_size=BATCH_SIZE)
INPUT_SIZE = dados["input_size"]
log_nota(f"input_size = {INPUT_SIZE} | batch_size = {BATCH_SIZE}")


## 12. Modelo de classificação — inadimplência

### 12.1 Arquitetura

In [ ]:
modelo_clf = criar_mlp_classificacao_padrao(INPUT_SIZE).to(DEVICE)
print(modelo_clf)


### 12.2 Função de perda, otimizador e scheduler

O target de inadimplência é desbalanceado (a maioria dos empréstimos é
`Fully Paid`) — `pos_weight` no `BCEWithLogitsLoss` compensa isso
reponderando a classe minoritária no cálculo da perda, o mesmo ajuste já
usado nas iterações anteriores deste projeto.

In [ ]:
n_neg = df_treino.filter(pl.col("target_classificacao") == 0).height
n_pos = df_treino.filter(pl.col("target_classificacao") == 1).height
pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
log_nota(f"Classes de treino — negativos: {n_neg} | positivos: {n_pos} | pos_weight: {pos_weight.item():.3f}")

loss_fn_clf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
otimizador_clf = torch.optim.AdamW(modelo_clf.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler_clf = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador_clf, mode="min", factor=0.5, patience=3)


### 12.3 Treino

In [ ]:
CHECKPOINT_CLF = os.path.join(DIR_CHECKPOINTS, f"mlp_classificacao_{RUN_ID}.pt")

historico_clf = treinar_modelo(
    modelo=modelo_clf,
    loader_treino=dados["classificacao"]["train"],
    loader_val=dados["classificacao"]["val"],
    loss_fn=loss_fn_clf,
    otimizador=otimizador_clf,
    epocas=50,
    device=DEVICE,
    checkpoint_path=CHECKPOINT_CLF,
    scheduler=scheduler_clf,
    nome_run=f"classificacao_{RUN_ID}",
    paciencia_early_stopping=5,
)


### 12.4 Diagnóstico de treino

Recarrega os pesos do melhor checkpoint (menor `val_loss`, não necessariamente
o da última época, por causa do early stopping) antes de qualquer avaliação
ou inspeção de ativações.

In [ ]:
modelo_clf = carregar_melhor_modelo(modelo_clf, CHECKPOINT_CLF, device=DEVICE)

plot_curvas_loss(historico_clf, "classificacao", salvar_dir=DIR_REPORT_ASSETS)
plot_gradient_norm(historico_clf, "classificacao", salvar_dir=DIR_REPORT_ASSETS)


In [ ]:
# Percentual de ativações em zero por camada oculta — mapeia neurônios
# mortos usando um batch do conjunto de validação.
capturar_distribuicao_ativacoes(modelo_clf, dados["classificacao"]["val"], DEVICE, "classificacao")


### 12.5 Avaliação final (conjunto de teste)

In [ ]:
metricas_clf = avaliar_classificacao(modelo_clf, dados["classificacao"]["test"], DEVICE, "classificacao")
metricas_clf


## 13. Modelo de regressão — taxa de juros (`int_rate`)

### 13.1 Arquitetura

In [ ]:
modelo_reg = criar_mlp_regressao_padrao(INPUT_SIZE).to(DEVICE)
print(modelo_reg)


### 13.2 Função de perda, otimizador e scheduler

In [ ]:
loss_fn_reg = nn.MSELoss()
otimizador_reg = torch.optim.AdamW(modelo_reg.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler_reg = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador_reg, mode="min", factor=0.5, patience=3)


### 13.3 Treino

In [ ]:
CHECKPOINT_REG = os.path.join(DIR_CHECKPOINTS, f"mlp_regressao_{RUN_ID}.pt")

historico_reg = treinar_modelo(
    modelo=modelo_reg,
    loader_treino=dados["regressao"]["train"],
    loader_val=dados["regressao"]["val"],
    loss_fn=loss_fn_reg,
    otimizador=otimizador_reg,
    epocas=50,
    device=DEVICE,
    checkpoint_path=CHECKPOINT_REG,
    scheduler=scheduler_reg,
    nome_run=f"regressao_{RUN_ID}",
    paciencia_early_stopping=5,
)


### 13.4 Diagnóstico de treino

In [ ]:
modelo_reg = carregar_melhor_modelo(modelo_reg, CHECKPOINT_REG, device=DEVICE)

plot_curvas_loss(historico_reg, "regressao", salvar_dir=DIR_REPORT_ASSETS)
plot_gradient_norm(historico_reg, "regressao", salvar_dir=DIR_REPORT_ASSETS)


In [ ]:
capturar_distribuicao_ativacoes(modelo_reg, dados["regressao"]["val"], DEVICE, "regressao")


### 13.5 Avaliação final (conjunto de teste)

In [ ]:
metricas_reg = avaliar_regressao(modelo_reg, dados["regressao"]["test"], DEVICE, "regressao")
metricas_reg


## 14. Baselines lineares (scikit-learn)

Referência não trivial para justificar a complexidade extra dos MLPs:
regressão logística (classificação) e regressão linear (regressão), ambas
treinadas sobre os mesmos arrays de treino/teste já padronizados — extraídos
com a mesma `extrair_arrays` usada internamente por `preparar_dataloaders`,
então não há divergência de features entre o baseline e a rede neural.

`treinar_baseline_classificacao`/`treinar_baseline_regressao` não chamam
`log_etapa` com o resultado final (só `log_nota` no início) — por isso os
dois `log_etapa` explícitos logo abaixo, para os baselines aparecerem no
relatório exportado na etapa 16 com o mesmo nível de detalhe que os MLPs.

In [ ]:
X_treino_np, y_treino_clf_np, y_treino_reg_np = extrair_arrays(df_treino, dados_features := [
    c for c in df_treino.columns if c not in {"id", "issue_d", "target_classificacao", "target_regressao"}
])
X_teste_np, y_teste_clf_np, y_teste_reg_np = extrair_arrays(df_teste, dados_features)


In [ ]:
baseline_clf = treinar_baseline_classificacao(X_treino_np, y_treino_clf_np, X_teste_np, y_teste_clf_np)
log_etapa("Baseline — Regressão Logística (classificação)", baseline_clf)
baseline_clf


In [ ]:
baseline_reg = treinar_baseline_regressao(X_treino_np, y_treino_reg_np, X_teste_np, y_teste_reg_np)
log_etapa("Baseline — Regressão Linear (regressão)", baseline_reg)
baseline_reg


## 15. Registro de experimentos

Consolida hiperparâmetros e métricas de cada modelo (MLP e baseline) na
tabela global de experimentos, para comparação lado a lado na exportação
final.

In [ ]:
registrar_experimento(
    nome=f"mlp_classificacao_{RUN_ID}",
    tipo="classificacao",
    hiperparametros={
        "camadas_ocultas": [128, 64],
        "dropout": 0.2,
        "batch_size": BATCH_SIZE,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "pos_weight": pos_weight.item(),
    },
    metricas=metricas_clf,
    notas="Comparar com baseline_clf (regressão logística) abaixo.",
)

registrar_experimento(
    nome="baseline_logistica",
    tipo="classificacao",
    hiperparametros={"class_weight": "balanced", "max_iter": 1000},
    metricas=baseline_clf,
    notas=f"Baseline de referência para mlp_classificacao_{RUN_ID}.",
)


In [ ]:
registrar_experimento(
    nome=f"mlp_regressao_{RUN_ID}",
    tipo="regressao",
    hiperparametros={
        "camadas_ocultas": [128, 64, 32],
        "dropout": 0.2,
        "batch_size": BATCH_SIZE,
        "lr": 1e-3,
        "weight_decay": 1e-2,
    },
    metricas=metricas_reg,
    notas="Comparar com baseline_reg (regressão linear) abaixo.",
)

registrar_experimento(
    nome="baseline_linear",
    tipo="regressao",
    hiperparametros={},
    metricas=baseline_reg,
    notas=f"Baseline de referência para mlp_regressao_{RUN_ID}.",
)


## 16. Exportação de artefatos

Grava o log markdown (todo `log_etapa`/`log_nota` acumulado desde o início
do notebook) e a tabela de experimentos em JSON, ambos dentro de
`DIR_REPORT_ASSETS` — persistidos no Drive, sobrevivem a um reinício de
runtime.

In [ ]:
exportar_log(path=os.path.join(DIR_REPORT_ASSETS, f"log_execucao_{RUN_ID}.md"))
exportar_experimentos(path_json=os.path.join(DIR_REPORT_ASSETS, "experimentos.json"))


## 17. TensorBoard (opcional)

Inspeciona interativamente as curvas de loss, learning rate e norma do
gradiente de ambos os treinos, direto no notebook.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DIR_RUNS}


## Resumo desta execução

- `RUN_ID`: identifica os checkpoints (`outputs/checkpoints/`) e os runs do
  TensorBoard (`runs/`) gerados por esta rodada específica.
- `metricas_clf` / `baseline_clf`: comparação MLP vs. regressão logística na
  tarefa de classificação de inadimplência.
- `metricas_reg` / `baseline_reg`: comparação MLP vs. regressão linear na
  tarefa de regressão da taxa de juros.
- Log completo e tabela de experimentos exportados em `outputs/report_assets/`.

Para uma nova rodada de treino (outros hiperparâmetros, por exemplo), volte
à seção 4 para gerar um novo `RUN_ID` antes de re-executar as seções 12/13 —
assim o run anterior não é sobrescrito.